# UNet++ RD — results and visual analysis

**Completed 50-epoch benchmark · frozen epoch 48 · official test n = 1,077**

This CPU-only notebook verifies the saved result archives, displays publication-style tables, and exports PNG/PDF figures and CSV tables. It performs **no training, threshold tuning, or checkpoint selection**. RD is the positive class; the decision threshold remains 0.5.

Run from the repository in VS Code with a local Python kernel containing pandas, numpy, matplotlib, scikit-learn and IPython. The two small result archives must already be under `outputs/kaggle_results/`; see `docs/UNETPP_RESULTS.md` for retrieval. No GPU, video files or model libraries are needed. Generated artifacts and the executed notebook are ignored outputs.

In [1]:
from pathlib import Path
import sys
import json
from IPython.display import display, Markdown

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / 'src/rd_ultrasound_ml/unetpp_reporting.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open this notebook from inside the rd-ultrasound-ml repository.')
sys.path.insert(0, str(repo_root / 'src'))
from rd_ultrasound_ml.unetpp_reporting import load_verified_run, make_tables, create_figures, display_table
report_dir = repo_root / 'outputs/unetpp_analysis'
run = load_verified_run(repo_root)
tables = make_tables(run)
print(json.dumps(run['audit'], indent=2))


ModuleNotFoundError: No module named 'numpy'

## 1. Official partitions and class imbalance

The released clip assignments are unchanged. Non-RD combines Normal and PVD; RD combines macula-intact and macula-detached clips. There is no clip overlap across partitions. The paper states that patient IDs were removed and clip-level splits may share patients. Thus these results do not establish unseen-patient generalization.

In [ ]:
display_table(tables['split_summary'], {'RD fraction': lambda x: f'{x:.2%}'})
figures = create_figures(run, report_dir)
display(figures['split_balance'])


## 2. Engineering pilots — validation only

The two five-epoch custom-CNN pilots used 256 training clips and 16 sampled frames. Positive-class weighting improved ranking scores in that small run but **both variants missed all 40 validation RD clips at threshold 0.5**. These values are transcribed from the executed pilot notebook (see experiment log), rather than reverified from a downloaded pilot archive. They are not test results or a controlled architecture comparison with full-data UNet++.

In [ ]:
display_table(tables['pilot_summary'])


## 3. Learning over 50 epochs

Training used the full 3,873 training clips, 96-frame inputs, Adam, effective batch 48 and mixed precision. Checkpoint selection was fixed as **highest validation accuracy, earliest tie**. Epoch 48 was selected; epoch 49 tied. The vertical line marks the already frozen selection. The early flat accuracy concealed zero RD sensitivity. One AMP update was skipped in epoch 35; the audit records it.

The weighted pilot and full UNet++ run change multiple factors; do not attribute the improvement to architecture alone.

In [ ]:
display_table(run['history'].loc[run['history'].epoch.isin([1,12,19,25,48,49,50]), ['epoch','train_loss','validation_loss','accuracy','sensitivity','specificity','average_precision']])
display(figures['learning_curves'])


## 4. Frozen validation and official-test metrics

Validation metrics describe checkpoint selection; official-test metrics describe the frozen checkpoint on held-out clips. Displaying both does not merge the partitions. An always-non-RD predictor scores 977/1,077 = 90.71% test accuracy and detects zero RD clips.

In [ ]:
display_table(tables['metric_summary'])
display(tables['test_confusion'])
display(figures['test_confusion'])


## 5. Ranking performance at the frozen operating point

The plotted dot is the original threshold 0.5. ROC and precision–recall curves are descriptive summaries, not a threshold search. Average precision (AP) is the non-interpolated sklearn statistic; it is not labeled trapezoidal PR-AUC. Scores have not been calibrated as clinical probabilities.

In [ ]:
display(figures['test_curves'])
display(figures['score_distribution'])


## 6. Test-to-test comparison with the ERDES paper

Source: [Ozkut et al., ERDES v2, Table 8 — Non-RD vs. RD, UNet++](https://arxiv.org/html/2508.04735v2). The source values are frozen author-reported numbers. Our run follows the released encoder-classifier, with disclosed Adam/code versus AdamW/text, decoder, mixed-precision and accumulation differences. These findings complete one feasible adapted benchmark, rather than exact numerical replication. The paper's 0.879 sensitivity does not correspond to an integer count out of the released 100 RD clips; aggregation/denominator details remain unresolved.

In [ ]:
display_table(tables['paper_comparison'], {'Our test': lambda x:f'{x:.2%}', 'Paper test':lambda x:f'{x:.2%}', 'Difference (percentage points)':lambda x:f'{x:+.2f}'})
display(figures['paper_comparison'])


## 7. Uncertainty and interpretation

The 95% Wilson intervals below treat clips as independent Bernoulli observations. They are **illustrative clip-level intervals**, not patient-clustered confidence intervals; unknown patient overlap weakens that assumption. A single split/seed and an exposed test partition limit what can be concluded. No significance or equivalence claim is made.

The practical finding is 86 detected and 14 missed RD clips, with 13 non-RD false alarms. These are dataset-label/model-prediction outcomes, not clinical validation.

In [ ]:
display(figures['metric_intervals'])


## 8. Exported artifacts and next work

Figures are saved as PNG and vector PDF, tables as CSV, and verification provenance as JSON under `outputs/unetpp_analysis/`. The source notebook is kept free of embedded experiment outputs; an executed copy is saved separately.

Use these results for the progress presentation. Define a subsequent controlled experiment using training/validation data and a written plan. Preserve this reference checkpoint and report; do not tune against official-test predictions or describe a reused test set as a fresh untouched holdout.

In [ ]:
print('Export directory:', report_dir)
for path in sorted(report_dir.iterdir()):
    print(path.name)
import matplotlib.pyplot as plt
plt.close('all')
